# ReverseBIT — Google Colab Training & Inference


## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil

DRIVE_PROJECT_PATH = '/content/drive/MyDrive/final-project'
# ────────────────────────────────────────────────────────────────────────────

COLAB_WORK_DIR = '/content/final-project'
os.makedirs(COLAB_WORK_DIR, exist_ok=True)
os.chdir(COLAB_WORK_DIR)

# Copy Python source files
for fname in [
    'reverse_blur_interpolation_transformer_training.py',
    'reverse_blur_interpolation_transformer_inferer.py'
]:
    src = os.path.join(DRIVE_PROJECT_PATH, fname)
    dst = os.path.join(COLAB_WORK_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'Copied: {fname}')
    else:
        print(f'WARNING: {fname} not found in {DRIVE_PROJECT_PATH}')

# Symlink dataset (avoids slow copying of large image folders)
dataset_src = os.path.join(DRIVE_PROJECT_PATH, 'dataset')
dataset_dst = os.path.join(COLAB_WORK_DIR, 'dataset')
if not os.path.exists(dataset_dst):
    if os.path.exists(dataset_src):
        os.symlink(dataset_src, dataset_dst)
        print('Dataset symlinked from Drive.')
    else:
        print(f'WARNING: dataset folder not found at {dataset_src}')
else:
    print('Dataset already linked.')

print(f'\nWorking directory: {os.getcwd()}')
print('Files present:', os.listdir('.'))

In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM           :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
import glob

train_scenes = sorted(os.listdir('dataset/train')) if os.path.exists('dataset/train') else []
test_scenes  = sorted(os.listdir('dataset/test'))  if os.path.exists('dataset/test')  else []

print(f'Train scenes ({len(train_scenes)}): {train_scenes}')
print(f'Test  scenes ({len(test_scenes)}):  {test_scenes}')

# Count frames in first train scene
if train_scenes:
    s = train_scenes[0]
    sharp_frames = glob.glob(f'dataset/train/{s}/sharp/*.png')
    blur_frames  = glob.glob(f'dataset/train/{s}/blur/*.png')
    print(f'\nScene "{s}" — sharp: {len(sharp_frames)} | blur: {len(blur_frames)}')

## 5. Configuring Training Hyperparameters

In [ ]:
# ── Edit these before training ───────────────────────────────────────────────
CONFIG = dict(
    epochs        = 30,
    batch_size    = 2,       # increase to 8 if GPU VRAM > 12 GB
    lr            = 1e-4,
    patch_size    = 192,
    num_workers   = 2,
    # Model capacity
    dim           = 64,      # 32 for quick test, 64 for full training
    num_heads     = 4,
    num_rstb      = 2,
    window_size   = 8,
)

# Checkpoint dir on Drive (so checkpoints survive Colab restarts)
CKPT_DIR_DRIVE = os.path.join(DRIVE_PROJECT_PATH, 'checkpoints')
CKPT_DIR_LOCAL = os.path.join(COLAB_WORK_DIR, 'checkpoints')
os.makedirs(CKPT_DIR_DRIVE, exist_ok=True)
os.makedirs(CKPT_DIR_LOCAL, exist_ok=True)
print('Config:', CONFIG)

## 6. Training the Model

In [ ]:
import math
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from reverse_blur_interpolation_transformer_training import (
    RBIDataset, create_model, SSIMLoss, compute_psnr, compute_ssim
)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Training on: {device}')

# ── Dataset ──────────────────────────────────────────────────────────────────
train_dataset = RBIDataset('dataset', split='train', patch_size=CONFIG['patch_size'])
val_dataset   = RBIDataset('dataset', split='test',  patch_size=CONFIG['patch_size'])
train_loader  = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                           shuffle=True,  num_workers=CONFIG['num_workers'], pin_memory=True)
val_loader    = DataLoader(val_dataset,   batch_size=CONFIG['batch_size'],
                           shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

# ── Model ─────────────────────────────────────────────────────────────────────
model = create_model(
    dim=CONFIG['dim'], num_heads=CONFIG['num_heads'],
    num_rstb_blocks=CONFIG['num_rstb'], window_size=CONFIG['window_size']
).to(device)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=CONFIG['epochs'], eta_min=1e-6)
l1_loss   = nn.L1Loss()
ssim_loss = SSIMLoss().to(device)

# ── Resume from checkpoint if available ──────────────────────────────────────
start_epoch = 0
best_psnr   = 0.0
resume_path = os.path.join(CKPT_DIR_LOCAL, 'best_model.pth')
drive_best  = os.path.join(CKPT_DIR_DRIVE, 'best_model.pth')
if not os.path.exists(resume_path) and os.path.exists(drive_best):
    shutil.copy2(drive_best, resume_path)
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    best_psnr   = ckpt.get('val_psnr', 0.0)
    start_epoch = ckpt.get('epoch', 0) + 1
    print(f'Resumed from epoch {start_epoch - 1} (best PSNR: {best_psnr:.2f} dB)')

# ── Training loop ─────────────────────────────────────────────────────────────
for epoch in range(start_epoch, CONFIG['epochs']):

    # Training
    model.train()
    epoch_loss = 0.0
    for batch_idx, (sharp_frames, blur_target) in enumerate(train_loader):
        sharp_frames = sharp_frames.to(device)
        blur_target  = blur_target.to(device)

        optimizer.zero_grad()
        blur_pred = model(sharp_frames).clamp(0, 1)
        loss = 0.8 * l1_loss(blur_pred, blur_target) + \
               0.2 * ssim_loss(blur_pred, blur_target)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

        if batch_idx % 50 == 0:
            print(f'  Epoch [{epoch}/{CONFIG["epochs"]}] '
                  f'Step [{batch_idx}/{len(train_loader)}] '
                  f'Loss: {loss.item():.4f}')

    # Validation
    model.eval()
    val_psnr_total = val_ssim_total = val_loss_total = 0.0
    with torch.no_grad():
        for sharp_frames, blur_target in val_loader:
            sharp_frames = sharp_frames.to(device)
            blur_target  = blur_target.to(device)
            blur_pred    = model(sharp_frames)
            blur_clamped = blur_pred.clamp(0, 1)
            val_loss_total += (0.8 * l1_loss(blur_pred, blur_target) +
                               0.2 * ssim_loss(blur_pred, blur_target)).item()
            val_psnr_total += compute_psnr(blur_clamped, blur_target)
            val_ssim_total += compute_ssim(blur_clamped, blur_target)

    avg_train = epoch_loss       / len(train_loader)
    avg_val   = val_loss_total   / len(val_loader)
    avg_psnr  = val_psnr_total   / len(val_loader)
    avg_ssim  = val_ssim_total   / len(val_loader)

    print(f'\nEpoch [{epoch}/{CONFIG["epochs"]}] Summary:')
    print(f'  Train Loss : {avg_train:.4f}')
    print(f'  Val Loss   : {avg_val:.4f}')
    print(f'  Val PSNR   : {avg_psnr:.2f} dB')
    print(f'  Val SSIM   : {avg_ssim:.4f}')
    print(f'  LR         : {scheduler.get_last_lr()[0]:.2e}\n')

    # Save epoch checkpoint locally
    ckpt_payload = {
        'epoch': epoch, 'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': avg_train, 'val_loss': avg_val,
        'val_psnr': avg_psnr, 'val_ssim': avg_ssim,
    }
    local_ckpt = os.path.join(CKPT_DIR_LOCAL, f'epoch_{epoch}.pth')
    torch.save(ckpt_payload, local_ckpt)

    # Save best model to both local and Drive
    if avg_psnr > best_psnr:
        best_psnr = avg_psnr
        best_payload = {'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'val_psnr': avg_psnr, 'val_ssim': avg_ssim}
        torch.save(best_payload, os.path.join(CKPT_DIR_LOCAL, 'best_model.pth'))
        torch.save(best_payload, os.path.join(CKPT_DIR_DRIVE, 'best_model.pth'))
        print(f'  ✓ New best model saved to Drive (PSNR: {best_psnr:.2f} dB)')

    # Also back up latest epoch to Drive
    shutil.copy2(local_ckpt, os.path.join(CKPT_DIR_DRIVE, f'epoch_{epoch}.pth'))

    scheduler.step()

print(f'\nTraining complete. Best PSNR: {best_psnr:.2f} dB')

## 7. Run Inference

In [ ]:
import glob
import torch
import torchvision.utils as vutils
from torchvision import transforms
from PIL import Image
from reverse_blur_interpolation_transformer_training import create_model

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Load model ────────────────────────────────────────────────────────────────
checkpoint_path = os.path.join(CKPT_DIR_LOCAL, 'best_model.pth')
model = create_model(
    dim=CONFIG['dim'], num_heads=CONFIG['num_heads'],
    num_rstb_blocks=CONFIG['num_rstb'], window_size=CONFIG['window_size']
).to(device)
state = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(state['model_state_dict'])
model.eval()
print(f'Model loaded (epoch {state["epoch"]}, PSNR {state["val_psnr"]:.2f} dB)')

# ── Load sharp window ─────────────────────────────────────────────────────────
INPUT_SHARP_DIR = 'dataset/train/scene_18/sharp'  # change if needed
NUM_FRAMES      = 9

to_tensor = transforms.ToTensor()
paths = sorted(glob.glob(os.path.join(INPUT_SHARP_DIR, '*.png')))
if len(paths) < NUM_FRAMES:
    raise ValueError(f'Need {NUM_FRAMES} frames, found {len(paths)}')

frames = torch.stack([to_tensor(Image.open(p).convert('RGB')) for p in paths[:NUM_FRAMES]])
sharp_frames = frames.unsqueeze(0).to(device)   # (1, N, 3, H, W)

# ── Model prediction ──────────────────────────────────────────────────────────
with torch.no_grad():
    blur_pred = model(sharp_frames).squeeze(0).clamp(0, 1).cpu()

baseline = sharp_frames.squeeze(0).cpu().mean(dim=0)   # naive average

# ── Save results ─────────────────────────────────────────────────────────────
RESULTS_LOCAL = os.path.join(COLAB_WORK_DIR, 'results')
RESULTS_DRIVE = os.path.join(DRIVE_PROJECT_PATH, 'results')
os.makedirs(RESULTS_LOCAL, exist_ok=True)
os.makedirs(RESULTS_DRIVE, exist_ok=True)

vutils.save_image(blur_pred, os.path.join(RESULTS_LOCAL, 'rbit_output.png'))
vutils.save_image(baseline,  os.path.join(RESULTS_LOCAL, 'naive_baseline.png'))

comparison = torch.stack([baseline, blur_pred], dim=0)
vutils.save_image(comparison, os.path.join(RESULTS_LOCAL, 'comparison.png'), nrow=2, padding=4)

# Copy to Drive
for f in ['rbit_output.png', 'naive_baseline.png', 'comparison.png']:
    shutil.copy2(os.path.join(RESULTS_LOCAL, f), os.path.join(RESULTS_DRIVE, f))

print('Results saved to Drive!')

## 8. Visualise Results

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
titles = ['Naive Average Baseline', 'ReverseBIT Output']
files  = ['naive_baseline.png', 'rbit_output.png']

for ax, title, fname in zip(axes, titles, files):
    img = mpimg.imread(os.path.join(RESULTS_LOCAL, fname))
    ax.imshow(img)
    ax.set_title(title, fontsize=14)
    ax.axis('off')

plt.suptitle('Blur Synthesis Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DRIVE, 'comparison_plot.png'), dpi=150)
plt.show()